# 03 — Feature Selection and Leakage Review



## 1. Imports


In [1]:
from pathlib import Path
import json
import pandas as pd


## 2. Project Root & Paths


In [2]:
def find_project_root() -> Path:
    cur = Path.cwd().resolve()
    for p in [cur] + list(cur.parents):
        if (p / "data").exists() and (p / "experiments").exists():
            return p
    raise RuntimeError("Could not locate project root.")

PROJECT_ROOT = find_project_root()
IN_CSV = PROJECT_ROOT / "data" / "interim" / "features" / "feature_engineered_dataset.csv"
OUT_JSON = PROJECT_ROOT / "data" / "processed" / "features" / "selected_features.json"
OUT_JSON.parent.mkdir(parents=True, exist_ok=True)


## 3. Load Dataset


In [3]:
df = pd.read_csv(IN_CSV)
print("Columns:", len(df.columns))


Columns: 74


## 4. Define Leakage & PII Exclusions


In [4]:
LEAKAGE_COLS = ["approved_financial_assistance", "remaining_medical_expenses", "case_study_narrative", "social_economic_assessment", "assessment_notes", "social_worker_assessment_rating", "relevant_social_worker_findings", "status", "existing_financial_assistance"]
PII_AND_ADMIN = ["address", "philhealth_no", "doctor", "ap_consultant", "referred_by", "follow_up_date", "date_available", "date_registered", "quotation_particular", "quotation_amount", "quotation_remarks", "other_sources_of_financial_support", "interview_responses", "other_eligibility_certifications", "estimated_treatment_expenses"]


## 5. Filter Selected Predictors


In [5]:
excluded = [c for c in LEAKAGE_COLS + PII_AND_ADMIN if c in df.columns]
selected = [c for c in df.columns if c not in set(excluded + ["eligibility"])]


## 6. Save Selected Feature Manifest


In [6]:
OUT_JSON.write_text(json.dumps({"selected_features": selected, "excluded_features": excluded}, indent=2), encoding="utf-8")
print(f"Excluded: {len(excluded)} | Selected: {len(selected)}")


Excluded: 24 | Selected: 49


### Feature Exclusion Policy & Zero-Variance Audit
Load `models/phase_05/feature_audit_table.csv` and verify zero-variance predictors.

In [7]:
audit_path = PROJECT_ROOT / "models" / "phase_05" / "feature_audit_table.csv"
if audit_path.exists():
    audit_df = pd.read_csv(audit_path)
    display(audit_df["decision"].value_counts().reset_index(name="count"))
    display(audit_df[audit_df["decision"] == "KEEP"].head(15))

,decision,count
0,KEEP,36
1,EXCLUDE,33
2,EXCLUDE (Target),1
3,EXCLUDE (Provenance),1


,column_name,data_meaning,pii_identifier_status,prediction_time_available,leakage_risk,decision,justification
0,charity_patient_no,Intake demographic / socioeconomic / clinical ...,No,Yes,Low,KEEP,Valid intake predictor
4,age,Intake demographic / socioeconomic / clinical ...,No,Yes,Low,KEEP,Valid intake predictor
5,sex,Intake demographic / socioeconomic / clinical ...,No,Yes,Low,KEEP,Valid intake predictor
6,marital_status,Intake demographic / socioeconomic / clinical ...,No,Yes,Low,KEEP,Valid intake predictor
7,contact_no,Intake demographic / socioeconomic / clinical ...,No,Yes,Low,KEEP,Valid intake predictor
9,birthday,Intake demographic / socioeconomic / clinical ...,No,Yes,Low,KEEP,Valid intake predictor
10,number_of_dependents,Intake demographic / socioeconomic / clinical ...,No,Yes,Low,KEEP,Valid intake predictor
11,monthly_family_income,Intake demographic / socioeconomic / clinical ...,No,Yes,Low,KEEP,Valid intake predictor
12,monthly_household_obligations_expenses,Intake demographic / socioeconomic / clinical ...,No,Yes,Low,KEEP,Valid intake predictor
13,employment_status,Intake demographic / socioeconomic / clinical ...,No,Yes,Low,KEEP,Valid intake predictor
